In [10]:
import requests
import pandas as pd
import time
from dotenv import load_dotenv

# Show all columns
pd.set_option('display.max_columns', None)
import os
import json

from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote

import threading
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sqlalchemy.dialects.mysql import insert as mysql_insert

import hashlib

In [ ]:
# === Configuration ===
BASE_URL = "https://ebuzima.moh.gov.rw"

API_KEY = 
API_SECRET = 

In [4]:
ENDPOINT = f"{BASE_URL}/api/resource/DocType"

# === Set the Authorization header ===
headers = {
    "Authorization": f"token {API_KEY}:{API_SECRET}",
    "Accept": "application/json"
}

# === Initialize ===
all_doctypes = []
limit_start = 0
limit_page_length = 100

# === Pagination Loop ===
while True:
    params = {
        "limit_start": limit_start,
        "limit_page_length": limit_page_length
    }

    response = requests.get(ENDPOINT, headers=headers, params=params)

    if response.status_code != 200:
        print(f"❌ Error fetching data: {response.status_code}\n{response.text}")
        break

    data = response.json().get("data", [])
    if not data:
        break  # No more data

    all_doctypes.extend(data)
    limit_start += limit_page_length

In [5]:
df_doc = pd.DataFrame(all_doctypes)

### Meta Data Retrieval

## Health Facilities

In [6]:
# === Configuration ===
BASE_URL = "https://ebuzima.moh.gov.rw"

headers = {
    "Authorization": f"token {API_KEY}:{API_SECRET}",
    "Content-Type": "application/json"
}

# === Filter for patients created in July–August 2024 ===
start_date = "2023-07-01 00:00:00"
end_date = "2026-09-02 00:00:00"
filters = [
    ["creation", ">=", start_date],
    ["creation", "<", end_date]
]

# === Pagination setup ===
page_size = 100
offset = 0
patients_full = []

# === Loop to fetch all patients in bulk ===
while True:
    payload = {
        "doctype": "Company",
        "filters": filters,
        "fields": ["*"],  # Replace with specific fields if needed
        "limit_start": offset,
        "limit_page_length": page_size
    }

    response = requests.post(
        f"{BASE_URL}/api/method/frappe.client.get_list",
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print(f"❌ Failed to fetch batch at offset {offset}: {response.status_code} - {response.text}")
        break

    batch = response.json().get("message", [])
    if not batch:
        print("✅ All records retrieved.")
        break

    print(f"📦 Retrieved {len(batch)} full patient records (offset {offset})")
    patients_full.extend(batch)
    offset += page_size

df_fac = pd.DataFrame(patients_full)

print(f"🎉 Total patients retrieved: {len(patients_full)}")

📦 Retrieved 100 full patient records (offset 0)
📦 Retrieved 100 full patient records (offset 100)
📦 Retrieved 100 full patient records (offset 200)
📦 Retrieved 36 full patient records (offset 300)
✅ All records retrieved.
🎉 Total patients retrieved: 336


In [8]:
df_fac.to_csv("../data/metadata/ebuzima_facilities.csv", index=False)

## Data Retrieval

### Antenatal Care

In [9]:
# === Configuration ===
DOCTYPE    = "Antenatal Care"
START_DATE = "2025-04-01 00:00:00"
END_DATE   = "2026-04-01 00:00:00"
FIELDS     = ["*"]
PAGE_SIZE  = 500

filters = [
    ["creation", ">=", START_DATE],
    ["creation", "<",  END_DATE]
]

# === Thread-local session (each thread gets its own) ===
thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({
            "Authorization": f"token {API_KEY}:{API_SECRET}",
            "Content-Type": "application/json"
        })
        thread_local.session = s
    return thread_local.session

# === Step 1: Get total count ===
s = get_session()
count_resp = s.post(
    f"{BASE_URL}/api/method/frappe.client.get_count",
    json={"doctype": DOCTYPE, "filters": filters}
)
total = count_resp.json().get("message", 0)
print(f"Total records to fetch: {total:,}")

# === Step 2: Fetcher ===
def fetch_page(offset, retries=3):
    s = get_session()
    payload = {
        "doctype": DOCTYPE,
        "filters": filters,
        "fields": FIELDS,
        "limit_start": offset,
        "limit_page_length": PAGE_SIZE
    }
    for attempt in range(retries):
        try:
            r = s.post(
                f"{BASE_URL}/api/method/frappe.client.get_list",
                json=payload,
                timeout=60
            )
            if r.status_code == 200:
                return r.json().get("message", [])
            time.sleep(1 * (attempt + 1))
        except Exception:
            time.sleep(1 * (attempt + 1))
    print(f"❌ Giving up at offset {offset}")
    return []

# === Step 3: Parallel fetch ===
offsets = range(0, total, PAGE_SIZE)
patients_full = []

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(fetch_page, off): off for off in offsets}
    for i, future in enumerate(as_completed(futures)):
        batch = future.result()
        patients_full.extend(batch)
        if i % 100 == 0:
            print(f"📦 Progress: {len(patients_full):,} / {total:,}")

# === Step 4: DataFrame ===
df_anc = pd.DataFrame(patients_full)

print(f"🎉 Total retrieved: {len(df_anc):,} rows × {len(df_anc.columns)} columns")

Total records to fetch: 181,648
📦 Progress: 500 / 181,648
📦 Progress: 50,500 / 181,648
📦 Progress: 100,500 / 181,648
📦 Progress: 150,500 / 181,648
🎉 Total retrieved: 181,648 rows × 121 columns


In [11]:
def hash_value(val):
    if pd.isna(val) or val == "":
        return val
    return hashlib.sha256(str(val).encode()).hexdigest()

cols_to_hash = ['patient_name', 'phone_number', 'practitioner', 'practitioner_name', 'partner_name', 'owner', 'modified_by']

for col in cols_to_hash:
    df_anc[col] = df_anc[col].apply(hash_value)

print("Done")

Done


In [12]:
df_anc.to_csv("../data/raw/ebuzima_anc_april_2025_mar_2026.csv", index=False)

In [13]:
# === Configuration ===
DOCTYPE    = "Antenatal Care Followup"
PARENT     = "Antenatal Care"
start_date = "2025-04-01 00:00:00"
end_date   = "2026-04-01 00:00:00"
FIELDS     = ["*"]
PAGE_SIZE  = 5000

filters = [
    ["followup_date", ">=", start_date],
    ["followup_date", "<",  end_date]
]

# or_filters = [
#     ["creation", ">=", START_DATE],
#     ["modified", ">=", START_DATE]
# ]

# === Thread-local session ===
thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({
            "Authorization": f"token {API_KEY}:{API_SECRET}"
        })
        thread_local.session = s
    return thread_local.session

# === Step 1: Get total count using get_list aggregate ===
s = get_session()
count_payload = {
    "doctype": DOCTYPE,
    "parent": PARENT,
    "filters": json.dumps(filters),
    # "or_filters": json.dumps(or_filters),
    "fields": json.dumps(["count(name) as total"]),
    "limit_page_length": 1
}

count_resp = s.post(
    f"{BASE_URL}/api/method/frappe.client.get_list",
    data=count_payload,
    timeout=60
)
count_resp.raise_for_status()

count_result = count_resp.json().get("message", [])
total = int(count_result[0]["total"]) if count_result else 0
print(f"Total records to fetch: {total:,}")

# === Step 2: Fetcher ===
def fetch_page(offset, retries=3):
    s = get_session()
    payload = {
        "doctype": DOCTYPE,
        "parent": PARENT,
        "filters": json.dumps(filters),
        # "or_filters": json.dumps(or_filters),
        "fields": json.dumps(FIELDS),
        "limit_start": offset,
        "limit_page_length": PAGE_SIZE
    }

    for attempt in range(retries):
        try:
            r = s.post(
                f"{BASE_URL}/api/method/frappe.client.get_list",
                data=payload,
                timeout=120
            )
            if r.status_code == 200:
                return r.json().get("message", [])
            time.sleep(attempt + 1)
        except Exception:
            time.sleep(attempt + 1)

    print(f"❌ Giving up at offset {offset}")
    return []

# === Step 3: Parallel fetch ===
offsets = range(0, total, PAGE_SIZE)
all_rows = []

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(fetch_page, off): off for off in offsets}
    for i, future in enumerate(as_completed(futures), 1):
        batch = future.result()
        all_rows.extend(batch)

        if i % 20 == 0 or i == 1:
            print(f"📦 Progress: {len(all_rows):,} / {total:,}")

# === Step 4: DataFrame ===
df_anc_followup = pd.DataFrame(all_rows)

print(f"🎉 Total retrieved: {len(df_anc_followup):,} rows × {len(df_anc_followup.columns)} columns")

Total records to fetch: 357,482
📦 Progress: 5,000 / 357,482
📦 Progress: 100,000 / 357,482
📦 Progress: 200,000 / 357,482
📦 Progress: 300,000 / 357,482
🎉 Total retrieved: 357,482 rows × 117 columns


In [14]:
cols_to_hash_foll = ['modified_by', 'owner', 'practitioner', 'practitioner_name']

for col in cols_to_hash_foll:
    df_anc_followup[col] = df_anc_followup[col].apply(hash_value)

print("Done")

Done


In [16]:
df_anc_followup.to_csv("../data/raw/ebuzima_anc_followup_april_2025_mar_2026.csv", index=False)

### Maternity

In [17]:
# === Configuration ===
BASE_URL = os.getenv("BASE_URL", "https://ebuzima.moh.gov.rw")

headers = {
    "Authorization": f"token {API_KEY}:{API_SECRET}",
    "Content-Type": "application/json"
}

start_date = "2025-04-01 00:00:00"
end_date   = "2026-04-01 00:00:00"
filters = [
    ["creation", ">=", start_date],
    ["creation", "<",  end_date]
]
fields = ["*"]
page_size = 500  # was 100

# Step 1: Get total record count
count_resp = requests.post(
    f"{BASE_URL}/api/method/frappe.client.get_count",
    headers=headers,
    json={"doctype": "Maternity Register", "filters": filters}
)
total = count_resp.json().get("message", 0)
print(f"Total records to fetch: {total}")

# Step 2: Define per-page fetcher
def fetch_page(offset):
    payload = {
        "doctype": "Maternity Register",
        "filters": filters,
        "fields": fields,
        "limit_start": offset,
        "limit_page_length": page_size
    }
    r = requests.post(
        f"{BASE_URL}/api/method/frappe.client.get_list",
        headers=headers,
        json=payload,
        timeout=60
    )
    if r.status_code != 200:
        print(f"❌ Failed at offset {offset}: {r.status_code}")
        return []
    return r.json().get("message", [])

# Step 3: Fetch all pages in parallel
offsets = range(0, total, page_size)
patients_full = []

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_page, off): off for off in offsets}
    for future in as_completed(futures):
        batch = future.result()
        patients_full.extend(batch)
        print(f"📦 {len(batch)} records (offset {futures[future]})")

df_mat= pd.DataFrame(patients_full)
print(f"🎉 Total retrieved: {len(patients_full)}")

Total records to fetch: 59011
📦 500 records (offset 0)
📦 500 records (offset 1500)
📦 500 records (offset 500)
📦 500 records (offset 4500)
📦 500 records (offset 2000)
📦 500 records (offset 1000)
📦 500 records (offset 3000)
📦 500 records (offset 3500)
📦 500 records (offset 2500)
📦 500 records (offset 4000)
📦 500 records (offset 5000)
📦 500 records (offset 5500)
📦 500 records (offset 6000)
📦 500 records (offset 6500)
📦 500 records (offset 7500)
📦 500 records (offset 8500)
📦 500 records (offset 8000)
📦 500 records (offset 7000)
📦 500 records (offset 9000)
📦 500 records (offset 9500)
📦 500 records (offset 10000)
📦 500 records (offset 11500)
📦 500 records (offset 11000)
📦 500 records (offset 10500)
📦 500 records (offset 12000)
📦 500 records (offset 13000)
📦 500 records (offset 12500)
📦 500 records (offset 13500)
📦 500 records (offset 14000)
📦 500 records (offset 14500)
📦 500 records (offset 15000)
📦 500 records (offset 15500)
📦 500 records (offset 16000)
📦 500 records (offset 16500)
📦 500 re

In [21]:
cols_to_hash_mat = ['modified_by', 'owner', 'head_of_family_id', 'head_of_family_names', 'practitioner', 'practitioner_names', '_seen', 'patient_names']

for col in cols_to_hash_mat:
    df_mat[col] = df_mat[col].apply(hash_value)

print("Done")

Done


In [23]:
df_mat.to_csv("../data/raw/ebuzima_mat_april_oct_2025_mar_2026.csv", index=False)

### Newborn

In [19]:
# === Configuration ===
DOCTYPE    = "Newborn"
PARENT     = "Maternity Register"
START_DATE = "2025-04-01 00:00:00"
END_DATE   = "2026-04-01 00:00:00"
FIELDS     = ["*"]
PAGE_SIZE  = 5000

filters = [
    ["creation", "<", END_DATE],
    ["modified", "<", END_DATE]
]

or_filters = [
    ["creation", ">=", START_DATE],
    ["modified", ">=", START_DATE]
]

# === Thread-local session ===
thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({
            "Authorization": f"token {API_KEY}:{API_SECRET}"
        })
        thread_local.session = s
    return thread_local.session

# === Step 1: Get total count using get_list aggregate ===
s = get_session()
count_payload = {
    "doctype": DOCTYPE,
    "parent": PARENT,
    "filters": json.dumps(filters),
    "or_filters": json.dumps(or_filters),
    "fields": json.dumps(["count(name) as total"]),
    "limit_page_length": 1
}

count_resp = s.post(
    f"{BASE_URL}/api/method/frappe.client.get_list",
    data=count_payload,
    timeout=60
)
count_resp.raise_for_status()

count_result = count_resp.json().get("message", [])
total = int(count_result[0]["total"]) if count_result else 0
print(f"Total records to fetch: {total:,}")

# === Step 2: Fetcher ===
def fetch_page(offset, retries=3):
    s = get_session()
    payload = {
        "doctype": DOCTYPE,
        "parent": PARENT,
        "filters": json.dumps(filters),
        "or_filters": json.dumps(or_filters),
        "fields": json.dumps(FIELDS),
        "limit_start": offset,
        "limit_page_length": PAGE_SIZE
    }

    for attempt in range(retries):
        try:
            r = s.post(
                f"{BASE_URL}/api/method/frappe.client.get_list",
                data=payload,
                timeout=120
            )
            if r.status_code == 200:
                return r.json().get("message", [])
            time.sleep(attempt + 1)
        except Exception:
            time.sleep(attempt + 1)

    print(f"❌ Giving up at offset {offset}")
    return []

# === Step 3: Parallel fetch ===
offsets = range(0, total, PAGE_SIZE)
all_rows = []

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(fetch_page, off): off for off in offsets}
    for i, future in enumerate(as_completed(futures), 1):
        batch = future.result()
        all_rows.extend(batch)

        if i % 20 == 0 or i == 1:
            print(f"📦 Progress: {len(all_rows):,} / {total:,}")

# === Step 4: DataFrame ===
df_newborn = pd.DataFrame(all_rows)

print(f"🎉 Total retrieved: {len(df_newborn):,} rows × {len(df_newborn.columns)} columns")
# print(df_lab_pre.head())

Total records to fetch: 21,259
📦 Progress: 5,000 / 21,259
🎉 Total retrieved: 21,259 rows × 39 columns


In [25]:
cols_to_hash_newborn = ['modified_by', 'owner', 'names', 'patient_name']

for col in cols_to_hash_newborn:
    df_newborn[col] = df_newborn[col].apply(hash_value)

print("Done")


Done


In [26]:
df_newborn.to_csv("../data/raw/ebuzima_newborn_april_2025_mar_2026.csv", index=False)